# llm-finetune-inference-lab on Colab Pro+ (A100 40GB)

End-to-end run: data preparation, QLoRA SFT, DPO, evaluation, quantized export and vLLM serving.
All heavy lifting lives in the repository CLI; this notebook only orchestrates it, so a dropped session resumes from Drive checkpoints.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/dataeclipse/llm-finetune-inference-lab.git
%cd llm-finetune-inference-lab
!curl -LsSf https://astral.sh/uv/install.sh | sh
!~/.local/bin/uv sync --extra train

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
!~/.local/bin/uv run lab data prepare profile=colab_a100
!~/.local/bin/uv run lab data stats profile=colab_a100

## SFT (QLoRA, resumes automatically from the last Drive checkpoint)

In [ ]:
!~/.local/bin/uv run lab train sft profile=colab_a100

## Baseline evaluation, DPO pair mining and DPO

Pair mining samples the SFT model through a vLLM endpoint and keeps only generations that fail execution checks; gold SQL becomes `chosen`.

In [ ]:
!~/.local/bin/uv run lab eval run profile=colab_a100 --model-path Qwen/Qwen3-8B
!~/.local/bin/uv run lab eval run profile=colab_a100

In [ ]:
!~/.local/bin/uv sync --extra train --extra serve
!~/.local/bin/uv run lab export merge profile=colab_a100 --output-dir /content/sft-merged
import subprocess, time
server = subprocess.Popen(['/root/.local/bin/uv', 'run', 'vllm', 'serve', '/content/sft-merged', '--served-model-name', 'sft', '--max-model-len', '4096'])
time.sleep(180)

In [ ]:
!~/.local/bin/uv run lab train pairs profile=colab_a100 --base-url http://localhost:8000/v1 --model sft
server.terminate()
!~/.local/bin/uv run lab train dpo profile=colab_a100

## Final evaluation, merge and quantized exports

In [ ]:
!~/.local/bin/uv run lab eval run profile=colab_a100 --model-path $(~/.local/bin/uv run lab config show profile=colab_a100 | grep 'output_dir' | head -2 | tail -1)
!~/.local/bin/uv run lab export merge profile=colab_a100
!~/.local/bin/uv sync --extra train --extra quant
!~/.local/bin/uv run lab export awq profile=colab_a100
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
%env LLAMA_CPP_DIR=/content/llama.cpp
!~/.local/bin/uv run lab export gguf profile=colab_a100

## Serve the merged model with vLLM and benchmark

In [ ]:
server = subprocess.Popen(['/root/.local/bin/uv', 'run', 'lab', 'serve', 'vllm', 'profile=colab_a100'])
time.sleep(180)
!~/.local/bin/uv run lab bench latency --base-url http://localhost:8000/v1 --concurrency 8 --requests 64
!~/.local/bin/uv run lab bench gpu --duration 60 --interval 5

In [ ]:
!~/.local/bin/uv sync --extra train --extra ui
!~/.local/bin/uv run python ui/app.py --base-url http://localhost:8000/v1 --model qwen3-8b-sql